# Laboratorio 6. Transfer Learning y Fine-Tuning sobre CIFAR-10

CC3092 Deep Learning y Sistemas Inteligentes. Brandon Werner Rivera Cabrera 23088

Se comparan tres estrategias para clasificar CIFAR-10: una CNN entrenada desde cero, VGG-16 preentrenada en ImageNet como extractor de características y VGG-16 con fine-tuning de sus bloques superiores. Cada modelo se itera al menos tres veces, se mide su costo computacional y se evalúa una única vez sobre el conjunto de test.

**Cómo se ejecuta.** El notebook corre de principio a fin sin intervención (`jupyter nbconvert --execute`). Todos los resultados quedan en `results/` y todas las figuras en `figs/`. El preprocesamiento y la augmentation se hacen directamente en la GPU sobre tensores, lo que evita los procesos de `DataLoader` (problemáticos en Windows dentro de Jupyter) y hace que el redimensionamiento a la entrada de VGG-16 no sea un cuello de botella.

## 1. Configuración

In [ ]:
import os, time, json, copy, random, platform, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import datasets, models
from torchvision.models import VGG16_Weights
from torchvision.transforms import v2 as T
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix
from torchinfo import summary

warnings.filterwarnings("ignore", category=UserWarning)
SMOKE = os.environ.get("LAB6_SMOKE") == "1"       # prueba rápida con pocos datos (solo para verificar que todo corre)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
VGG_RES = 64 if SMOKE else 128                      # resolución de entrada para los dos modelos VGG-16
BATCH_CNN, BATCH_VGG, BATCH_EVAL = 256, 128, 500

def fijar_semilla(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
fijar_semilla()

torch.backends.cudnn.benchmark = True
USE_AMP = DEVICE.type == "cuda"
AMP_DTYPE = torch.bfloat16 if (USE_AMP and torch.cuda.is_bf16_supported()) else torch.float16
autocast = lambda: torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP)

os.makedirs("figs", exist_ok=True); os.makedirs("results", exist_ok=True); os.makedirs("checkpoints", exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 160, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 9})

def sincronizar():
    if DEVICE.type == "cuda": torch.cuda.synchronize()

entorno = dict(
    gpu=torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "sin GPU",
    cpu=platform.processor() or platform.machine(), sistema=platform.platform(),
    python=platform.python_version(), torch=torch.__version__, torchvision=torchvision.__version__,
    precision=str(AMP_DTYPE) if USE_AMP else "float32", vgg_res=VGG_RES, smoke=SMOKE,
)
print(json.dumps(entorno, indent=2, ensure_ascii=False))

## 2. Dataset y exploración

In [ ]:
train_full = datasets.CIFAR10(root="data", train=True, download=True)
test_set = datasets.CIFAR10(root="data", train=False, download=True)
CLASES = train_full.classes
X_all, y_all = train_full.data, np.array(train_full.targets)      # (50000, 32, 32, 3) uint8
X_test_np, y_test_np = test_set.data, np.array(test_set.targets)  # (10000, 32, 32, 3) uint8

print(f"Entrenamiento: {X_all.shape[0]:,} imágenes | Test: {X_test_np.shape[0]:,} imágenes | Clases: {len(CLASES)}")
print(f"Resolución: {X_all.shape[1]}x{X_all.shape[2]} píxeles, {X_all.shape[3]} canales (RGB), tipo {X_all.dtype}")
conteo = pd.DataFrame({"clase": CLASES, "train": np.bincount(y_all, minlength=10), "test": np.bincount(y_test_np, minlength=10)})
display(conteo.set_index("clase").T)
print("Clases balanceadas:", bool((conteo.train == conteo.train.iloc[0]).all() and (conteo.test == conteo.test.iloc[0]).all()))

### División estratificada 45 000 / 5 000
El conjunto de entrenamiento oficial se divide en entrenamiento y validación manteniendo 500 imágenes de cada clase en validación. El conjunto de test oficial queda reservado y se usa una sola vez, al final, con el mejor modelo de cada estrategia.

In [ ]:
idx_tr, idx_val = train_test_split(np.arange(len(y_all)), test_size=5000, stratify=y_all, random_state=SEED)
if SMOKE:  # solo para la prueba rápida
    idx_tr, idx_val = idx_tr[:600], idx_val[:200]
    X_test_np, y_test_np = X_test_np[:300], y_test_np[:300]
print("train:", len(idx_tr), "| val:", len(idx_val), "| test:", len(y_test_np))
print("imágenes por clase en validación:", np.bincount(y_all[idx_val], minlength=10))

# media y desviación estándar por canal, calculadas SOLO con la partición de entrenamiento
x_tr_float = X_all[idx_tr].astype(np.float32) / 255.0
CIFAR_MEAN = x_tr_float.mean(axis=(0, 1, 2)).tolist()
CIFAR_STD = x_tr_float.std(axis=(0, 1, 2)).tolist()
del x_tr_float
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
stats = pd.DataFrame({"CIFAR-10 media": CIFAR_MEAN, "ImageNet media": IMAGENET_MEAN,
                      "CIFAR-10 std": CIFAR_STD, "ImageNet std": IMAGENET_STD}, index=["R", "G", "B"]).round(4)
display(stats)

Las medias de CIFAR-10 son parecidas a las de ImageNet, con un poco más de azul relativo por la cantidad de cielo y agua en las clases de aviones y barcos. Las desviaciones estándar son algo mayores que las de ImageNet. Por eso la CNN desde cero se normaliza con las estadísticas propias del conjunto de entrenamiento, mientras que VGG-16 se normaliza con las de ImageNet: sus filtros aprendieron a esperar entradas en esa escala y cambiarla desplazaría todas las activaciones de la red.

In [ ]:
fig, axes = plt.subplots(10, 5, figsize=(5.2, 10.5))
rng = np.random.default_rng(SEED)
for c in range(10):
    ids = rng.choice(np.where(y_all == c)[0], 5, replace=False)
    for j, i in enumerate(ids):
        ax = axes[c, j]; ax.imshow(X_all[i]); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        for s in ax.spines.values(): s.set_visible(False)
    axes[c, 0].set_ylabel(CLASES[c], rotation=0, ha="right", va="center", fontsize=9)
plt.suptitle("Cinco ejemplos por clase", y=0.995)
plt.tight_layout(); plt.savefig("figs/ejemplos_por_clase.png", bbox_inches="tight"); plt.show()

**Pares que se espera que sean difíciles.** Gato y perro son el par más difícil: comparten forma, textura de pelaje y poses, y a 32×32 los rasgos que los distinguen ocupan unos pocos píxeles. Le siguen automóvil y camión, que comparten ruedas, ventanas y fondos de carretera; venado y caballo, cuadrúpedos de colores similares sobre pasto; y avión con pájaro o barco, porque los tres aparecen como una silueta sobre un fondo uniforme de cielo o agua.

## 3. Preprocesamiento y data augmentation

In [ ]:
# Pipelines "de referencia" con torchvision (documentan exactamente qué se aplica a cada modelo)
pipeline_cnn_train = T.Compose([T.ToImage(), T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(),
                                T.ToDtype(torch.float32, scale=True), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
pipeline_cnn_eval = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
pipeline_vgg_train = T.Compose([T.ToImage(), T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(),
                                T.Resize(VGG_RES), T.ToDtype(torch.float32, scale=True), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
pipeline_vgg_eval = T.Compose([T.ToImage(), T.Resize(VGG_RES), T.ToDtype(torch.float32, scale=True),
                               T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
print("Transformación oficial de los pesos de ImageNet (weights.transforms()):")
print(VGG16_Weights.IMAGENET1K_V1.transforms())

`weights.transforms()` redimensiona a 256, recorta el centro a 224 y normaliza con las estadísticas de ImageNet; está pensada para fotografías grandes. En CIFAR-10 el objeto ocupa toda la imagen de 32×32, así que un recorte central eliminaría parte del objeto: se conserva la normalización de ImageNet y se reemplaza el resize+crop por un redimensionamiento directo a la resolución de entrada elegida.

**Implementación en GPU.** Las mismas operaciones de los pipelines anteriores se aplican por lote en la GPU: el recorte aleatorio con relleno de 4 píxeles y el volteo horizontal se implementan como una transformación afín por imagen (`affine_grid` + `grid_sample`), que en el caso de VGG-16 produce directamente la imagen redimensionada a la resolución de entrada; luego se normaliza. Con desplazamientos enteros el remuestreo es exacto, equivalente a `RandomCrop(32, padding=4)`.

In [ ]:
def a_tensor(X):  # (N, 32, 32, 3) uint8 -> (N, 3, 32, 32) uint8 en el dispositivo
    return torch.from_numpy(np.ascontiguousarray(X)).permute(0, 3, 1, 2).contiguous().to(DEVICE)

X_tr, y_tr = a_tensor(X_all[idx_tr]), torch.from_numpy(y_all[idx_tr]).to(DEVICE)
X_val, y_val = a_tensor(X_all[idx_val]), torch.from_numpy(y_all[idx_val]).to(DEVICE)
X_te, y_te = a_tensor(X_test_np), torch.from_numpy(y_test_np).to(DEVICE)

def preparar(xb, size, mean, std, augment):
    # xb: uint8 (B,3,32,32). Devuelve float normalizado (B,3,size,size) en channels_last.
    x = xb.float().div_(255.0)
    B = x.shape[0]
    if augment:
        theta = torch.zeros(B, 2, 3, device=x.device)
        theta[:, 0, 0] = torch.where(torch.rand(B, device=x.device) < 0.5, -1.0, 1.0)  # volteo horizontal
        theta[:, 1, 1] = 1.0
        theta[:, :, 2] = torch.randint(-4, 5, (B, 2), device=x.device).float() * (2.0 / 32)  # traslación de ±4 píxeles
        grid = F.affine_grid(theta, (B, 3, size, size), align_corners=False)
        x = F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    elif size != x.shape[-1]:
        x = F.interpolate(x, size=(size, size), mode="bilinear", align_corners=False)
    m = torch.tensor(mean, device=x.device).view(1, 3, 1, 1); s = torch.tensor(std, device=x.device).view(1, 3, 1, 1)
    return ((x - m) / s).contiguous(memory_format=torch.channels_last)

def lotes(X, y, batch, shuffle, size, mean, std, augment):
    n = len(y)
    idx = torch.randperm(n, device=X.device) if shuffle else torch.arange(n, device=X.device)
    for i in range(0, n, batch):
        b = idx[i:i + batch]
        yield preparar(X[b], size, mean, std, augment), y[b]

# Visualización de los pipelines: original, CNN con augmentation, VGG-16 con augmentation y redimensionada
muestra = X_val[:6]
desnorm = lambda x, m, s: (x.float().cpu() * torch.tensor(s).view(1, 3, 1, 1) + torch.tensor(m).view(1, 3, 1, 1)).clamp(0, 1)
filas = [("original 32×32", muestra.float().cpu() / 255),
         ("CNN: augmentation 32×32", desnorm(preparar(muestra, 32, CIFAR_MEAN, CIFAR_STD, True), CIFAR_MEAN, CIFAR_STD)),
         (f"VGG-16: augmentation {VGG_RES}×{VGG_RES}", desnorm(preparar(muestra, VGG_RES, IMAGENET_MEAN, IMAGENET_STD, True), IMAGENET_MEAN, IMAGENET_STD))]
fig, axes = plt.subplots(3, 6, figsize=(8, 4.4))
for r, (tit, imgs) in enumerate(filas):
    for j in range(6):
        axes[r, j].imshow(imgs[j].permute(1, 2, 0).numpy()); axes[r, j].axis("off")
    axes[r, 0].set_title(tit, loc="left", fontsize=9)
plt.tight_layout(); plt.savefig("figs/pipelines.png", bbox_inches="tight"); plt.show()

**Píxeles y cómputo al aumentar la resolución.** Una imagen de 32×32 tiene 1 024 píxeles por canal y una de 224×224 tiene 50 176: 49 veces más. En una red convolucional el costo de cada capa es proporcional al área de sus mapas de activación, así que el cómputo crece prácticamente en la misma proporción que el número de píxeles. La celda siguiente lo mide para VGG-16. Por eso en este laboratorio los dos modelos VGG-16 usan 128×128: conserva gran parte de la utilidad de las características preentrenadas, porque los objetos quedan a una escala más cercana a la de ImageNet, y cuesta alrededor de tres veces menos que 224×224.

**Data augmentation.** Se usan recorte aleatorio con relleno de 4 píxeles y volteo horizontal. Ambas transformaciones producen imágenes que siguen siendo ejemplos válidos de su clase, porque un avión desplazado unos píxeles o reflejado de izquierda a derecha sigue siendo un avión, y obligan a la red a no depender de la posición exacta del objeto. No se usa volteo vertical ni rotaciones grandes porque generarían imágenes poco realistas, como barcos de cabeza. La augmentation se aplica solo al entrenamiento: validación y test deben medir el desempeño sobre las imágenes tal como son, de forma determinista y comparable entre épocas y modelos; aumentarlas agregaría ruido a la métrica sin representar datos reales.

## 4. Investigación: herramientas de transfer learning en PyTorch

- **`torchvision.models.vgg16` y `VGG16_Weights`**: construye la arquitectura VGG-16; con `weights=VGG16_Weights.IMAGENET1K_V1` descarga los pesos entrenados en ImageNet (71.6 % top-1). El enum también expone `weights.transforms()`, el preprocesamiento con el que se entrenaron, y `weights.meta` con categorías, métricas y costo.
- **Estructura del modelo**: `model.features` es la parte convolucional, 13 capas conv 3×3 en 5 bloques separados por max pooling; `model.avgpool` es un `nn.AdaptiveAvgPool2d((7, 7))` que lleva cualquier mapa final a 7×7, y por eso la red acepta resoluciones distintas de 224; `model.classifier` son tres capas densas 25 088→4 096→4 096→1 000 con ReLU y dropout, y se reemplaza la última por una de 10 salidas.
- **`requires_grad`**: cada parámetro tiene este atributo; si es `False` no se calcula su gradiente y el optimizador no lo modifica. Se congela un bloque completo con `modulo.requires_grad_(False)` y se descongela con `True`.
- **`model.train()`, `model.eval()` y `torch.no_grad()`**: `train()` y `eval()` cambian el comportamiento de capas como dropout y batch normalization; `no_grad()` desactiva el registro del grafo de autograd, ahorrando memoria y tiempo en evaluación. Son independientes: para evaluar se usan las dos.
- **Grupos de parámetros**: el optimizador acepta una lista de diccionarios `{"params": ..., "lr": ...}`, lo que permite un learning rate menor para el backbone preentrenado que para el clasificador nuevo.
- **`torchinfo.summary`**: recorre el modelo con una entrada de ejemplo y reporta por capa la forma de salida, los parámetros totales y entrenables y las multiplicaciones-sumas.
- **`torch.cuda.max_memory_allocated()` y `torch.cuda.synchronize()`**: el primero devuelve el pico de memoria reservada por tensores desde el último `reset_peak_memory_stats()`; el segundo espera a que la GPU termine, necesario para medir tiempos porque las operaciones CUDA son asíncronas.
- **MACs y FLOPs**: una MAC es una multiplicación seguida de una suma; un FLOP es una operación de punto flotante. Una MAC equivale a dos FLOPs, así que FLOPs ≈ 2 × MACs.
- **Parámetros totales y entrenables**: los totales son todos los pesos del modelo; los entrenables son los que tienen `requires_grad=True` y el optimizador actualiza. En transfer learning los totales definen memoria y costo de inferencia, mientras que los entrenables definen el costo del backward y el riesgo de sobreajuste.

In [ ]:
vgg_demo = models.vgg16(weights=None)
vgg_demo.classifier[6] = nn.Linear(4096, 10)
print(vgg_demo.avgpool)
print(vgg_demo.classifier)
BLOQUES = {1: (0, 5), 2: (5, 10), 3: (10, 17), 4: (17, 24), 5: (24, 31)}  # índices de model.features de cada bloque
for b, (i, j) in BLOQUES.items():
    capas = [type(l).__name__ for l in vgg_demo.features[i:j]]
    print(f"bloque {b}: features[{i}:{j}] ->", capas)

def contar_parametros(m):
    return sum(p.numel() for p in m.parameters()), sum(p.numel() for p in m.parameters() if p.requires_grad)

def contar_macs(model, res):
    # MACs por imagen de cada Conv2d y Linear (hooks) -> total y la parte entrenable
    macs, hooks = {}, []
    def hook(mod, inp, out):
        if isinstance(mod, nn.Conv2d):
            macs[mod] = (out.numel() // out.shape[0]) * (mod.in_channels // mod.groups) * mod.kernel_size[0] * mod.kernel_size[1]
        elif isinstance(mod, nn.Linear):
            macs[mod] = mod.in_features * mod.out_features
    for mod in model.modules():
        if isinstance(mod, (nn.Conv2d, nn.Linear)): hooks.append(mod.register_forward_hook(hook))
    estado = model.training; model.eval()
    with torch.no_grad(): model(torch.zeros(1, 3, res, res, device=next(model.parameters()).device))
    model.train(estado)
    for h in hooks: h.remove()
    total = sum(macs.values())
    entrenable = sum(v for mod, v in macs.items() if any(p.requires_grad for p in mod.parameters()))
    return total, entrenable

pix = []
for r in [32, 64, 112, 128, 224]:
    t, _ = contar_macs(vgg_demo, r)
    pix.append(dict(resolucion=f"{r}×{r}", pixeles=r * r, GMACs=round(t / 1e9, 3), GFLOPs=round(2 * t / 1e9, 3)))
pix = pd.DataFrame(pix); pix["relativo a 32×32"] = (pix.GMACs / pix.GMACs.iloc[0]).round(1)
display(pix)
pix.to_csv("results/costo_por_resolucion.csv", index=False)
summary(vgg_demo, input_size=(1, 3, VGG_RES, VGG_RES), depth=1, col_names=("output_size", "num_params", "mult_adds"), verbose=1)
del vgg_demo

## 5. Modelos

In [ ]:
class CNN(nn.Module):
    # Bloques [conv3x3-BN-ReLU] x 2 + MaxPool; cabeza con global average pooling
    def __init__(self, anchos=(64, 128, 256), dropout=0.0):
        super().__init__()
        capas, c_in = [], 3
        for c in anchos:
            capas += [nn.Conv2d(c_in, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True),
                      nn.Conv2d(c, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True),
                      nn.MaxPool2d(2)]
            if dropout > 0: capas.append(nn.Dropout2d(dropout / 2))
            c_in = c
        self.features = nn.Sequential(*capas)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(dropout), nn.Linear(c_in, 10))
    def forward(self, x):
        return self.head(self.features(x))

def construir_vgg(cfg):
    # modo "fe": backbone congelado. modo "ft": se descongelan los bloques de cfg["bloques"]
    try:
        m = models.vgg16(weights=VGG16_Weights.IMAGENET1K_V1)
    except Exception as e:
        raise RuntimeError("No se pudieron descargar los pesos de ImageNet de VGG-16") from e
    m.classifier[6] = nn.Linear(4096, 10)
    m.requires_grad_(False)
    if cfg["modo"] == "fe" and cfg.get("entrenar") == "ultima":
        m.classifier[6].requires_grad_(True)
    else:
        m.classifier.requires_grad_(True)
    if cfg["modo"] == "ft":
        inicio = BLOQUES[min(cfg["bloques"])][0]
        m.features[inicio:].requires_grad_(True)
    return m

def construir(cfg):
    m = CNN(cfg["anchos"], cfg["dropout"]) if cfg["familia"] == "cnn" else construir_vgg(cfg)
    return m.to(DEVICE, memory_format=torch.channels_last)

## 6. Entrenamiento y evaluación

In [ ]:
def normalizacion(cfg):
    return (32, CIFAR_MEAN, CIFAR_STD) if cfg["familia"] == "cnn" else (VGG_RES, IMAGENET_MEAN, IMAGENET_STD)

@torch.no_grad()
def evaluar(model, X, y, cfg):
    size, mean, std = normalizacion(cfg)
    model.eval(); perdida, preds = 0.0, []
    for xb, yb in lotes(X, y, BATCH_EVAL, False, size, mean, std, False):
        with autocast():
            logits = model(xb)
        perdida += F.cross_entropy(logits.float(), yb, reduction="sum").item()
        preds.append(logits.argmax(1))
    preds = torch.cat(preds).cpu().numpy(); yt = y.cpu().numpy()
    p, r, f1, _ = precision_recall_fscore_support(yt, preds, average="macro", zero_division=0)
    return dict(loss=perdida / len(yt), acc=accuracy_score(yt, preds), precision=p, recall=r, f1=f1), preds

def crear_optimizador(model, cfg):
    if cfg["familia"] == "vgg" and cfg["modo"] == "ft":
        backbone = [p for p in model.features.parameters() if p.requires_grad]
        grupos = [{"params": backbone, "lr": cfg["lr_backbone"]},
                  {"params": [p for p in model.classifier.parameters() if p.requires_grad], "lr": cfg["lr"]}]
    else:
        grupos = [{"params": [p for p in model.parameters() if p.requires_grad], "lr": cfg["lr"]}]
    if cfg["opt"] == "sgd":
        return torch.optim.SGD(grupos, momentum=0.9, nesterov=True, weight_decay=cfg.get("wd", 0.0))
    if cfg["opt"] == "adamw":
        return torch.optim.AdamW(grupos, weight_decay=cfg.get("wd", 0.0))
    return torch.optim.Adam(grupos)

def crear_scheduler(opt, cfg, pasos_totales):
    if cfg.get("sched") == "onecycle":
        return torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[g["lr"] for g in opt.param_groups],
                                                   total_steps=pasos_totales, pct_start=0.25)
    if cfg.get("sched") == "cosine":
        calentamiento = max(1, int(0.05 * pasos_totales))
        f = lambda s: (s + 1) / calentamiento if s < calentamiento else 0.5 * (1 + np.cos(np.pi * (s - calentamiento) / max(1, pasos_totales - calentamiento)))
        return torch.optim.lr_scheduler.LambdaLR(opt, f)
    return None

def entrenar(cfg, X, y, epochs=None, guardar_en=None, etiqueta=None):
    # Entrena una configuración. Devuelve (resumen, historial por época, estado del mejor epoch en CPU)
    fijar_semilla()
    epochs = epochs or cfg["epochs"]
    if SMOKE: epochs = 1
    model = construir(cfg)
    total, entrenables = contar_parametros(model)
    size, mean, std = normalizacion(cfg)
    batch = BATCH_CNN if cfg["familia"] == "cnn" else BATCH_VGG
    pasos = epochs * ((len(y) + batch - 1) // batch)
    opt = crear_optimizador(model, cfg); sched = crear_scheduler(opt, cfg, pasos)
    if DEVICE.type == "cuda": torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    historial, mejor, mejor_estado, t_acum = [], None, None, 0.0
    for ep in range(1, epochs + 1):
        model.train(); suma, n = 0.0, 0
        sincronizar(); t0 = time.perf_counter()
        for xb, yb in lotes(X, y, batch, True, size, mean, std, cfg["augment"]):
            with autocast():
                loss = F.cross_entropy(model(xb), yb)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            if sched is not None: sched.step()
            suma += loss.item() * len(yb); n += len(yb)
        sincronizar(); t_ep = time.perf_counter() - t0; t_acum += t_ep
        val, _ = evaluar(model, X_val, y_val, cfg)
        fila = dict(iteracion=etiqueta or cfg["nombre"], familia=cfg["familia_etq"], epoch=ep, train_loss=suma / n,
                    val_loss=val["loss"], val_acc=val["acc"], val_precision=val["precision"], val_recall=val["recall"],
                    val_f1=val["f1"], t_epoch=t_ep, t_acum=t_acum)
        historial.append(fila)
        print(f"  [{fila['iteracion']}] ep {ep:2d}/{epochs}  train {fila['train_loss']:.4f}  val {val['loss']:.4f}  "
              f"acc {val['acc']:.4f}  f1 {val['f1']:.4f}  ({t_ep:.1f} s)", flush=True)
        if mejor is None or val["f1"] > mejor["val_f1"]:
            mejor = fila
            mejor_estado = {k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()}
    pico = torch.cuda.max_memory_allocated() / 2**20 if DEVICE.type == "cuda" else float("nan")
    resumen = dict(iteracion=etiqueta or cfg["nombre"], familia=cfg["familia_etq"], descripcion=cfg["descripcion"],
                   params_totales=total, params_entrenables=entrenables, epochs=epochs, mejor_epoch=mejor["epoch"],
                   val_acc=mejor["val_acc"], val_precision=mejor["val_precision"], val_recall=mejor["val_recall"],
                   val_f1=mejor["val_f1"], val_loss=mejor["val_loss"], t_epoch_medio=t_acum / epochs, t_total=t_acum,
                   t_hasta_mejor=mejor["t_acum"], memoria_pico_mb=pico)
    if guardar_en: torch.save(mejor_estado, guardar_en)
    del model, opt
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    return resumen, historial, mejor_estado

## 7. Iteraciones

Se entrenan 10 iteraciones: 3 de la CNN desde cero, 3 de VGG-16 como extractor de características y 4 de VGG-16 con fine-tuning. Dentro de cada estrategia se cambia una cosa a la vez, y el fine-tuning varía tanto el número de bloques descongelados como el learning rate del backbone. En cada iteración se conserva el estado del epoch con mejor F1 macro en validación.

In [ ]:
BASE_CNN = dict(familia="cnn", familia_etq="CNN desde cero")
BASE_FE = dict(familia="vgg", modo="fe", familia_etq="VGG-16 feature extractor")
BASE_FT = dict(familia="vgg", modo="ft", familia_etq="VGG-16 fine-tuning")
ITERACIONES = [
    dict(BASE_CNN, nombre="CNN-1", descripcion="3 bloques (32, 64, 128), sin augmentation ni dropout, Adam 1e-3 constante",
         anchos=(32, 64, 128), dropout=0.0, augment=False, opt="adam", lr=1e-3, sched=None, epochs=20),
    dict(BASE_CNN, nombre="CNN-2", descripcion="3 bloques (64, 128, 256), augmentation, dropout 0.3, AdamW 1e-3 coseno",
         anchos=(64, 128, 256), dropout=0.3, augment=True, opt="adamw", lr=1e-3, wd=5e-4, sched="cosine", epochs=30),
    dict(BASE_CNN, nombre="CNN-3", descripcion="4 bloques (64 a 512), augmentation, dropout 0.3, SGD OneCycle 0.1",
         anchos=(64, 128, 256, 512), dropout=0.3, augment=True, opt="sgd", lr=0.1, wd=5e-4, sched="onecycle", epochs=40),
    dict(BASE_FE, nombre="FE-1", descripcion="solo la última capa entrenable, sin augmentation, Adam 1e-3",
         entrenar="ultima", augment=False, opt="adam", lr=1e-3, sched=None, epochs=6),
    dict(BASE_FE, nombre="FE-2", descripcion="clasificador completo entrenable, sin augmentation, Adam 1e-4",
         entrenar="clasificador", augment=False, opt="adam", lr=1e-4, sched=None, epochs=6),
    dict(BASE_FE, nombre="FE-3", descripcion="clasificador completo, augmentation, AdamW 1e-4 coseno",
         entrenar="clasificador", augment=True, opt="adamw", lr=1e-4, wd=1e-4, sched="cosine", epochs=10),
    dict(BASE_FT, nombre="FT-1", descripcion="bloque 5 descongelado, lr backbone 1e-5, clasificador 1e-4",
         bloques=(5,), augment=True, opt="adamw", lr=1e-4, lr_backbone=1e-5, wd=1e-4, sched="cosine", epochs=8),
    dict(BASE_FT, nombre="FT-2", descripcion="bloques 4 y 5 descongelados, lr backbone 1e-5, clasificador 1e-4",
         bloques=(4, 5), augment=True, opt="adamw", lr=1e-4, lr_backbone=1e-5, wd=1e-4, sched="cosine", epochs=8),
    dict(BASE_FT, nombre="FT-3", descripcion="bloques 3, 4 y 5 descongelados, lr backbone 1e-5, clasificador 1e-4",
         bloques=(3, 4, 5), augment=True, opt="adamw", lr=1e-4, lr_backbone=1e-5, wd=1e-4, sched="cosine", epochs=8),
    dict(BASE_FT, nombre="FT-4", descripcion="bloques 4 y 5 descongelados, lr backbone 5e-5, clasificador 1e-4",
         bloques=(4, 5), augment=True, opt="adamw", lr=1e-4, lr_backbone=5e-5, wd=1e-4, sched="cosine", epochs=8),
]
pd.DataFrame([{k: c.get(k) for k in ("nombre", "familia_etq", "descripcion", "epochs")} for c in ITERACIONES])

In [ ]:
resumenes, historiales, mejores = [], [], {}
for cfg in ITERACIONES:
    print(f"\n=== {cfg['nombre']}: {cfg['descripcion']}")
    res, hist_it, estado = entrenar(cfg, X_tr, y_tr)
    resumenes.append(res); historiales += hist_it
    fam = cfg["familia_etq"]
    if fam not in mejores or res["val_f1"] > mejores[fam]["resumen"]["val_f1"]:
        mejores[fam] = dict(cfg=cfg, resumen=res)
        torch.save(estado, f"checkpoints/mejor_{cfg['familia']}_{cfg.get('modo', 'cnn')}.pt")
    del estado
    pd.DataFrame(resumenes).to_csv("results/iteraciones.csv", index=False)
    pd.DataFrame(historiales).to_csv("results/historial.csv", index=False)

tabla_it = pd.DataFrame(resumenes)
cols = ["iteracion", "descripcion", "params_totales", "params_entrenables", "mejor_epoch", "val_acc", "val_precision",
        "val_recall", "val_f1", "t_epoch_medio", "t_total", "memoria_pico_mb"]
display(tabla_it[cols].round(4))

In [ ]:
# Curvas de pérdida de entrenamiento y validación de todas las iteraciones
hist = pd.DataFrame(historiales)
familias = list(dict.fromkeys(c["familia_etq"] for c in ITERACIONES))
ncol = max(sum(c["familia_etq"] == f for c in ITERACIONES) for f in familias)
fig, axes = plt.subplots(len(familias), ncol, figsize=(3.1 * ncol, 2.5 * len(familias)), squeeze=False)
for r, fam in enumerate(familias):
    its = [c["nombre"] for c in ITERACIONES if c["familia_etq"] == fam]
    for j in range(ncol):
        ax = axes[r, j]
        if j >= len(its): ax.axis("off"); continue
        h = hist[hist.iteracion == its[j]]
        ax.plot(h.epoch, h.train_loss, marker="o", ms=3, color="#2a78d6", label="entrenamiento")
        ax.plot(h.epoch, h.val_loss, marker="o", ms=3, color="#eb6834", label="validación")
        ax.set_title(its[j], fontsize=9, loc="left"); ax.set_xlabel("epoch")
        if j == 0: ax.set_ylabel(f"{fam}\npérdida", fontsize=8.5)
axes[0, 0].legend(fontsize=8, frameon=False)
plt.tight_layout(); plt.savefig("figs/curvas_perdida.png", bbox_inches="tight"); plt.show()

## 8. Evaluación final sobre test

La mejor iteración de cada estrategia, elegida por F1 macro en validación, se evalúa una única vez sobre las 10 000 imágenes de test.

In [ ]:
test_res, matrices, modelos_finales = {}, {}, {}
for fam, info in mejores.items():
    cfg = info["cfg"]
    model = construir(cfg)
    model.load_state_dict(torch.load(f"checkpoints/mejor_{cfg['familia']}_{cfg.get('modo', 'cnn')}.pt", map_location=DEVICE))
    met, preds = evaluar(model, X_te, y_te, cfg)
    test_res[fam] = dict(iteracion=cfg["nombre"], **{k: float(v) for k, v in met.items()})
    matrices[fam] = confusion_matrix(y_te.cpu().numpy(), preds, labels=list(range(10)))
    modelos_finales[fam] = model
    print(f"{fam:<26} ({cfg['nombre']}): acc {met['acc']:.4f}  precision {met['precision']:.4f}  recall {met['recall']:.4f}  f1 {met['f1']:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
for ax, (fam, cm) in zip(axes, matrices.items()):
    cmn = cm / cm.sum(1, keepdims=True)
    im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(10)); ax.set_yticks(range(10)); ax.grid(False)
    ax.set_xticklabels(CLASES, rotation=60, ha="right", fontsize=7.5); ax.set_yticklabels(CLASES, fontsize=7.5)
    for i in range(10):
        for j in range(10):
            if cm[i, j] > 0: ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=6.5, color="white" if cmn[i, j] > 0.5 else "black")
    ax.set_title(f"{fam} ({test_res[fam]['iteracion']})", fontsize=9.5)
    ax.set_xlabel("predicción"); ax.set_ylabel("clase real")
plt.tight_layout(); plt.savefig("figs/matrices_confusion.png", bbox_inches="tight"); plt.show()

confusiones = []
for fam, cm in matrices.items():
    m = cm.copy(); np.fill_diagonal(m, 0); sim = m + m.T
    pares = sorted(((sim[i, j], CLASES[i], CLASES[j]) for i in range(10) for j in range(i + 1, 10)), reverse=True)[:3]
    for n, a, b in pares:
        confusiones.append(dict(modelo=fam, par=f"{a} / {b}", errores=int(n)))
display(pd.DataFrame(confusiones))
json.dump(dict(test=test_res, matrices={k: v.tolist() for k, v in matrices.items()}, confusiones=confusiones, clases=CLASES),
          open("results/test.json", "w"), indent=2, ensure_ascii=False)

## 9. Experimento: efecto de la cantidad de datos

Las mejores configuraciones de las tres estrategias se entrenan otra vez con el 10 % del conjunto de entrenamiento, conservando el balance entre clases, con la misma configuración y el mismo número de epochs, y se evalúan sobre el mismo test.

In [ ]:
idx_10, _ = train_test_split(np.arange(len(y_tr)), train_size=0.10, stratify=y_tr.cpu().numpy(), random_state=SEED)
idx_10 = torch.from_numpy(idx_10).to(DEVICE)
print("imágenes en el 10 %:", len(idx_10), "| por clase:", torch.bincount(y_tr[idx_10], minlength=10).tolist())
datos_exp = []
for fam, info in mejores.items():
    cfg = info["cfg"]
    print(f"\n=== {fam} con 10 % de los datos ({cfg['nombre']})")
    res10, hist10, estado10 = entrenar(cfg, X_tr[idx_10], y_tr[idx_10], etiqueta=cfg["nombre"] + " (10 %)")
    model = construir(cfg); model.load_state_dict(estado10)
    met, _ = evaluar(model, X_te, y_te, cfg)
    datos_exp.append(dict(modelo=fam, iteracion=cfg["nombre"], test_acc_100=test_res[fam]["acc"], test_f1_100=test_res[fam]["f1"],
                          test_acc_10=met["acc"], test_f1_10=met["f1"], caida_f1=test_res[fam]["f1"] - met["f1"],
                          t_total_10=res10["t_total"]))
    del model, estado10
datos_exp = pd.DataFrame(datos_exp)
datos_exp.to_csv("results/datos_10pct.csv", index=False)
display(datos_exp.round(4))

fig, ax = plt.subplots(figsize=(6.5, 3.2))
xs = np.arange(len(datos_exp)); w = 0.36
b1 = ax.bar(xs - w / 2, datos_exp.test_f1_100, w, color="#2a78d6", label="100 % de los datos")
b2 = ax.bar(xs + w / 2, datos_exp.test_f1_10, w, color="#eb6834", label="10 % de los datos")
ax.bar_label(b1, fmt="%.3f", fontsize=8); ax.bar_label(b2, fmt="%.3f", fontsize=8)
ax.set_xticks(xs); ax.set_xticklabels(datos_exp.modelo, fontsize=8.5); ax.set_ylabel("F1 macro en test"); ax.set_ylim(0, 1.08)
ax.legend(frameon=False, fontsize=8.5, loc="upper left", ncol=2); ax.grid(axis="x", visible=False)
plt.tight_layout(); plt.savefig("figs/experimento_datos.png", bbox_inches="tight"); plt.show()

## 10. Comparación de rendimiento y recursos

In [ ]:
@torch.no_grad()
def latencia_ms(model, size, repeticiones=100, calentamiento=20):
    model.eval(); x = torch.randn(1, 3, size, size, device=DEVICE).contiguous(memory_format=torch.channels_last)
    if SMOKE: repeticiones, calentamiento = 3, 1
    for _ in range(calentamiento):
        with autocast(): model(x)
    sincronizar(); t0 = time.perf_counter()
    for _ in range(repeticiones):
        with autocast(): model(x)
    sincronizar()
    return (time.perf_counter() - t0) / repeticiones * 1000

@torch.no_grad()
def throughput(model, size, batch=256, repeticiones=20):
    model.eval(); x = torch.randn(batch, 3, size, size, device=DEVICE).contiguous(memory_format=torch.channels_last)
    if SMOKE: repeticiones, batch = 2, 8
    with autocast(): model(x)
    sincronizar(); t0 = time.perf_counter()
    for _ in range(repeticiones):
        with autocast(): model(x)
    sincronizar()
    return batch * repeticiones / (time.perf_counter() - t0)

n_train = len(y_tr)
filas = []
for fam, info in mejores.items():
    cfg, res = info["cfg"], info["resumen"]
    model = modelos_finales[fam]
    size = normalizacion(cfg)[0]
    info_ti = summary(model, input_size=(1, 3, size, size), verbose=0)
    macs_total, macs_entrenable = contar_macs(model, size)
    flops_inf = 2 * macs_total
    flops_train_img = 2 * macs_total + 2 * 2 * macs_entrenable   # forward completo + backward (≈2x forward) de la parte entrenable
    filas.append(dict(
        modelo=fam, iteracion=cfg["nombre"], resolucion=f"{size}×{size}",
        params_totales=res["params_totales"], params_entrenables=res["params_entrenables"],
        GMACs_inferencia=macs_total / 1e9, GMACs_torchinfo=info_ti.total_mult_adds / 1e9, GFLOPs_inferencia=flops_inf / 1e9,
        latencia_ms=latencia_ms(model, size), imagenes_por_s=throughput(model, size),
        t_epoch_s=res["t_epoch_medio"], epochs_hasta_mejor=res["mejor_epoch"], epochs=res["epochs"], t_total_s=res["t_total"],
        memoria_pico_mb=res["memoria_pico_mb"],
        GFLOPs_entrenamiento_por_imagen=flops_train_img / 1e9,
        PFLOPs_entrenamiento_total=flops_train_img * n_train * res["epochs"] / 1e15,
        PFLOPs_hasta_mejor=flops_train_img * n_train * res["mejor_epoch"] / 1e15,
        test_acc=test_res[fam]["acc"], test_precision=test_res[fam]["precision"], test_recall=test_res[fam]["recall"],
        test_f1=test_res[fam]["f1"],
        test_f1_10pct=float(datos_exp.loc[datos_exp.modelo == fam, "test_f1_10"].iloc[0]),
    ))
comparacion = pd.DataFrame(filas)
comparacion.to_csv("results/comparacion.csv", index=False)
json.dump(entorno, open("results/entorno.json", "w"), indent=2, ensure_ascii=False)
display(comparacion.set_index("modelo").T)

In [ ]:
colores = {"CNN desde cero": "#2a78d6", "VGG-16 feature extractor": "#eb6834", "VGG-16 fine-tuning": "#1baf7a"}
hist = pd.DataFrame(historiales)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.9))
for fam, info in mejores.items():
    h = hist[hist.iteracion == info["cfg"]["nombre"]]
    ax1.plot(h.t_acum / 60, h.val_acc, marker="o", ms=3.5, lw=2, color=colores[fam], label=f"{fam} ({info['cfg']['nombre']})")
ax1.set_xlabel("tiempo de entrenamiento acumulado (min)"); ax1.set_ylabel("accuracy en validación")
ax1.set_title("Accuracy de validación contra tiempo de entrenamiento", fontsize=9.5, loc="left")
ax1.legend(frameon=False, fontsize=8)
desfase = {"CNN desde cero": (8, -2), "VGG-16 feature extractor": (8, -16), "VGG-16 fine-tuning": (8, 12)}
for _, r in comparacion.iterrows():
    ax2.scatter(r.PFLOPs_entrenamiento_total, r.test_f1, s=70, color=colores[r.modelo], zorder=3)
    ax2.annotate(f"{r.modelo}, {r.GFLOPs_inferencia:.2f} GFLOPs por imagen", (r.PFLOPs_entrenamiento_total, r.test_f1),
                 xytext=desfase[r.modelo], textcoords="offset points", fontsize=8, va="center")
ax2.set_xscale("log"); ax2.set_xlabel("cómputo total de entrenamiento (PFLOPs, escala log)"); ax2.set_ylabel("F1 macro en test")
ax2.set_title("Desempeño en test contra costo computacional", fontsize=9.5, loc="left")
xmin, xmax = comparacion.PFLOPs_entrenamiento_total.min(), comparacion.PFLOPs_entrenamiento_total.max()
ax2.set_xlim(xmin / 2.5, xmax * 12)
plt.tight_layout(); plt.savefig("figs/rendimiento_vs_recursos.png", bbox_inches="tight"); plt.show()

## 11. Resumen

Los archivos de `results/` contienen todo lo necesario para el reporte: `iteraciones.csv` (una fila por iteración), `historial.csv` (una fila por epoch), `test.json` (métricas de test, matrices de confusión y pares más confundidos), `datos_10pct.csv`, `comparacion.csv` y `entorno.json` con el hardware. Las figuras están en `figs/`. Los pesos de los mejores modelos quedan en `checkpoints/` y no se suben al repositorio por su tamaño.

In [ ]:
print("Hardware:", entorno["gpu"], "|", entorno["cpu"])
display(comparacion[["modelo", "iteracion", "params_entrenables", "GFLOPs_inferencia", "latencia_ms", "t_total_s",
                     "memoria_pico_mb", "test_acc", "test_f1", "test_f1_10pct"]].round(4))